<h2> Random Forest Model: Surgical Data

Guiding questions: Can perioperative and intraoperative variables like existing diabetes and hypertension impact surgical outcomes like death, length of ICU stays, and general prolonged hospital stay?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('vitaldb_cleaned.csv')
df

,case_id,patient_id,admission_time_from_case_start_sec,discharge_time_from_case_start_sec,icu_days,in_hospital_death,age,sex,height,weight,...,intraoperative_midazolam,intraoperative_fentanyl,hospital_length_of_stay,had_icu_stay,prolonged_hospital_stay,age_numeric,emergency_operation_label,in_hospital_death_label,icu_stay_label,prolonged_hospital_stay_label
0,1,5955,-236220,627780,0,0,77,M,160.2,67.50,...,0.0,100,10.0,0,0,77.0,Non-emergency,Survived,No ICU stay,Not prolonged
1,2,2487,-221160,1506840,0,0,54,M,167.3,54.80,...,0.0,0,20.0,0,1,54.0,Non-emergency,Survived,No ICU stay,Prolonged stay
2,3,2861,-218640,40560,0,0,62,M,169.1,69.70,...,0.0,0,3.0,0,0,62.0,Non-emergency,Survived,No ICU stay,Not prolonged
3,4,1903,-201120,576480,1,0,74,M,160.6,53.00,...,0.0,100,9.0,1,0,74.0,Non-emergency,Survived,Had ICU stay,Not prolonged
4,5,4416,-67560,3734040,13,0,66,M,171.0,59.70,...,0.0,0,44.0,1,1,66.0,Emergency,Survived,Had ICU stay,Prolonged stay
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6383,6384,5583,-215340,648660,0,0,64,M,161.5,63.00,...,0.0,0,10.0,0,0,64.0,Non-emergency,Survived,No ICU stay,Not prolonged
6384,6385,2278,-225600,1675200,0,0,69,M,159.3,62.30,...,0.0,0,22.0,0,1,69.0,Non-emergency,Survived,No ICU stay,Prolonged stay
6385,6386,4045,-200460,836340,0,0,61,F,151.7,43.25,...,0.0,0,12.0,0,1,61.0,Non-emergency,Survived,No ICU stay,Prolonged stay
6386,6387,5230,-227760,377040,0,0,24,F,155.7,55.50,...,0.0,0,7.0,0,0,24.0,Non-emergency,Survived,No ICU stay,Not prolonged


In [2]:
df.columns

Index(['case_id', 'patient_id', 'admission_time_from_case_start_sec',
       'discharge_time_from_case_start_sec', 'icu_days', 'in_hospital_death',
       'age', 'sex', 'height', 'weight', 'bmi', 'asa_class',
       'emergency_operation', 'department', 'operation_type', 'diagnosis',
       'operation_name', 'surgical_approach', 'patient_position',
       'anesthesia_type', 'preoperative_hypertension', 'preoperative_diabetes',
       'preoperative_hemoglobin', 'preoperative_platelet_count',
       'preoperative_creatinine', 'intraoperative_estimated_blood_loss',
       'intraoperative_urine_output',
       'intraoperative_red_blood_cell_transfusion',
       'intraoperative_fresh_frozen_plasma_transfusion',
       'intraoperative_crystalloid', 'intraoperative_colloid',
       'intraoperative_propofol', 'intraoperative_midazolam',
       'intraoperative_fentanyl', 'hospital_length_of_stay', 'had_icu_stay',
       'prolonged_hospital_stay', 'age_numeric', 'emergency_operation_label',
     

In [3]:
df = df.drop(columns = ['emergency_operation_label', 'in_hospital_death_label', 'icu_stay_label','prolonged_hospital_stay_label', 'age'])
#these are the target variables. target variables need to be numeric so I can use scikit learn 
#also, if I keep it in the X matrix, these columns pose the possibility of multicollinearity because these are just relabelled versions of existing columns


In [4]:
df

,case_id,patient_id,admission_time_from_case_start_sec,discharge_time_from_case_start_sec,icu_days,in_hospital_death,sex,height,weight,bmi,...,intraoperative_fresh_frozen_plasma_transfusion,intraoperative_crystalloid,intraoperative_colloid,intraoperative_propofol,intraoperative_midazolam,intraoperative_fentanyl,hospital_length_of_stay,had_icu_stay,prolonged_hospital_stay,age_numeric
0,1,5955,-236220,627780,0,0,M,160.2,67.50,26.3,...,0,350.0,0,120,0.0,100,10.0,0,0,77.0
1,2,2487,-221160,1506840,0,0,M,167.3,54.80,19.6,...,0,800.0,0,150,0.0,0,20.0,0,1,54.0
2,3,2861,-218640,40560,0,0,M,169.1,69.70,24.4,...,0,200.0,0,0,0.0,0,3.0,0,0,62.0
3,4,1903,-201120,576480,1,0,M,160.6,53.00,20.5,...,0,2700.0,0,80,0.0,100,9.0,1,0,74.0
4,5,4416,-67560,3734040,13,0,M,171.0,59.70,20.4,...,8,7100.0,0,0,0.0,0,44.0,1,1,66.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6383,6384,5583,-215340,648660,0,0,M,161.5,63.00,24.2,...,0,550.0,0,150,0.0,0,10.0,0,0,64.0
6384,6385,2278,-225600,1675200,0,0,M,159.3,62.30,24.6,...,0,2500.0,0,100,0.0,0,22.0,0,1,69.0
6385,6386,4045,-200460,836340,0,0,F,151.7,43.25,18.8,...,0,1300.0,0,70,0.0,0,12.0,0,1,61.0
6386,6387,5230,-227760,377040,0,0,F,155.7,55.50,22.9,...,0,1150.0,0,120,0.0,0,7.0,0,0,24.0


<h3> In-Hospital Deaths

Random Forest Model

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

In [ ]:
#preparing training and test sets
X1 = df.drop(columns = 'in_hospital_death')
X_death = pd.get_dummies(X1, drop_first=True) #one hot encoding
y_death = df['in_hospital_death']

X_train_death, X_test_death, y_train_death, y_test_death = train_test_split(X_death, y_death, random_state=1)

In [17]:
rf_death = RandomForestClassifier(max_features = 0.5, 
                                  max_depth= 20, class_weight='balanced')
rf_death.fit(X_train_death, y_train_death)
rf_pred_death_train = rf_death.predict(X_train_death)
rf_pred_death_test = rf_death.predict(X_test_death)


rf_death_metrics_train = [accuracy_score(y_train_death, rf_pred_death_train), precision_score(y_train_death, rf_pred_death_train), 
                    recall_score(y_train_death, rf_pred_death_train), f1_score(y_train_death, rf_pred_death_train)]

rf_death_metrics_test = [accuracy_score(y_test_death, rf_pred_death_test), precision_score(y_test_death, rf_pred_death_test), 
                    recall_score(y_test_death, rf_pred_death_test), f1_score(y_test_death, rf_pred_death_test)]

rf_matrix = [rf_death_metrics_train, rf_death_metrics_test]

rf_results = pd.DataFrame(rf_matrix, index= ['train', 'test'], columns=['Accuracy', 'Precision', 'Recall', 'F1 Score'])
rf_results

,Accuracy,Precision,Recall,F1 Score
train,1.000000,1.000000,1.000000,1.0
test,0.992486,0.666667,0.285714,0.4


In [8]:
rf_feat_importance = pd.DataFrame({'Importance': rf_death.feature_importances_}, index = X_death.columns).sort_values('Importance', ascending=False)
rf_feat_importance = rf_feat_importance[rf_feat_importance['Importance'] > 0.001].sort_values('Importance', ascending=False)
rf_feat_importance
#removing columns that have a feature importance of less than 0.001

,Importance
hospital_length_of_stay,0.105264
discharge_time_from_case_start_sec,0.091061
case_id,0.089359
icu_days,0.059337
preoperative_creatinine,0.057414
...,...
diagnosis_End stage renal disease,0.001086
diagnosis_Cadaveric donor,0.001076
operation_name_Distal gastrectomy,0.001045
operation_name_Diagnostic laparoscopy,0.001012


In [9]:
cols_for_xg = list(rf_feat_importance.index)
#let's just put the columns that have feature importance in xgboost

In [ ]:
from xgboost import XGBClassifier
X_train_death_feat_imp = X_train_death[cols_for_xg]
X_test_death_feat_imp = X_test_death[cols_for_xg]


xgb_death = XGBClassifier(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=500,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42
)
xgb_death.fit(X_train_death_feat_imp, y_train_death, verbose=False)
xgb_death_preds_train = xgb_death.predict(X_train_death_feat_imp)
xgb_death_preds_test = xgb_death.predict(X_test_death_feat_imp)


xgb_death_metrics_train = [accuracy_score(y_train_death, xgb_death_preds_train), precision_score(y_train_death, xgb_death_preds_train), 
                    recall_score(y_train_death, xgb_death_preds_train), f1_score(y_train_death, xgb_death_preds_train)]

xgb_death_metrics_test = [accuracy_score(y_test_death, xgb_death_preds_test), precision_score(y_test_death, xgb_death_preds_test), 
                    recall_score(y_test_death, xgb_death_preds_test), f1_score(y_test_death, xgb_death_preds_test)]

xgb_matrix = [xgb_death_metrics_train, xgb_death_metrics_test]

xgb_results = pd.DataFrame(xgb_matrix, index= ['train', 'test'], columns=['Accuracy', 'Precision', 'Recall', 'F1 Score'])

xgb_results


,Accuracy,Precision,Recall,F1 Score
train,1.000000,1.00,1.000000,1.000000
test,0.993738,0.75,0.428571,0.545455


<h2> Prolonged Hospital Stay

In [37]:
X2 = df.drop(columns = 'prolonged_hospital_stay') #let's shorten prolonged hospital stay to phs
X_phs = pd.get_dummies(X2, drop_first=True) #one hot encoding
y_phs = df['prolonged_hospital_stay']

X_train_phs, X_test_phs, y_train_phs, y_test_phs = train_test_split(X_phs, y_phs, random_state=1)

In [38]:
rf_phs = RandomForestClassifier(random_state = 42)
rf_phs.fit(X_train_phs, y_train_phs)
rf_pred_phs_train = rf_phs.predict(X_train_phs)
rf_pred_phs_test = rf_phs.predict(X_test_phs)


rf_phs_metrics_train = [accuracy_score(y_train_phs, rf_pred_phs_train), precision_score(y_train_phs, rf_pred_phs_train), 
                    recall_score(y_train_phs, rf_pred_phs_train), f1_score(y_train_phs, rf_pred_phs_train)]

rf_phs_metrics_test = [accuracy_score(y_test_phs, rf_pred_phs_test), precision_score(y_test_phs, rf_pred_phs_test), 
                    recall_score(y_test_phs, rf_pred_phs_test), f1_score(y_test_phs, rf_pred_phs_test)]

rf_matrix = [rf_phs_metrics_train, rf_phs_metrics_test]

rf_results = pd.DataFrame(rf_matrix, index= ['train', 'test'], columns=['Accuracy', 'Precision', 'Recall', 'F1 Score'])
rf_results

,Accuracy,Precision,Recall,F1 Score
train,1.000000,1.000000,1.000000,1.000000
test,0.997495,0.993827,0.993827,0.993827
